In [ ]:
import pandas as pd
from modelens import RegressionAnalyzer, regression_models

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [ ]:
raw_data = pd.read_csv("dataset.csv")
df = raw_data.copy()

In [ ]:
raw_data.head()

In [ ]:
analyzer = RegressionAnalyzer(df=df, target="cnt")

In [ ]:
analyzer.info()

In [ ]:
target = "cnt"
drop_columns = ["casual", "registered", "dteday", "instant"]

for column in drop_columns:
    df = df.drop(columns=[column])

X = df.drop(columns=[target])
y = df[target]
features = df.columns.drop([target])

In [ ]:
analyzer.reinit(df=df,target=target)

#### Comaring Models

In [ ]:
models = regression_models()

# analyzer.compare_models(
#     models=models,
#     features=features,
#     export_html=True,
#     file_name="01-base-model-comparing",
# )

In [ ]:
selected_models = regression_models(
    smodels=[
        "CatBoost",
        "Gradient Boosting",
        "LightGBM",
    ]
)


# analyzer.compare_models(
#     models=selected_models,
#     features=features,
#     export_html=True,
#     file_name="02-selected-model",
# )

In [ ]:
param_grids = {
    "CatBoost": {
        "iterations": [300, 500, 1000],
        "depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1],
        "l2_leaf_reg": [1, 3, 5],
    },
    "LightGBM": {
        "n_estimators": [100, 300, 500],
        "learning_rate": [0.03, 0.1],
        "num_leaves": [15, 31, 63],
        "max_depth": [-1, 10],
        "min_child_samples": [10, 20],
        "colsample_bytree": [0.8, 1.0],
    },
    "Gradient Boosting": {
        "n_estimators": [100, 300, 500],
        "learning_rate": [0.03, 0.05, 0.1],
        "max_depth": [2, 3, 5],
        "min_samples_split": [2, 5],
        "min_samples_leaf": [1, 2, 4],
    },
}

# for name, model in selected_models.items():

#     analyzer.tune_model(
#         model=model,
#         features=features,
#         param_grid=param_grids[name],
#         export_html=True,
#         file_name=f"{name.lower().replace(' ', '-')}",
#     )

In [ ]:
_, suspicious_features = analyzer.correlation(features=features)

In [ ]:
analyzer.vif(features=features)

In [ ]:
from lightgbm import LGBMRegressor

selected_models["LightGBM"] = LGBMRegressor(
    colsample_bytree=0.8,
    max_depth=10,
    min_child_samples=10,
    learning_rate=0.1,
    n_estimators=100,
    num_leaves=15,
)

In [ ]:
# for name, model in selected_models.items():
#     analyzer.evaluate_single_feature_removal(
#         model=model,
#         features=features,
#         export_html=True,
#         file_name=f"{name.lower().replace(' ', '-')}",
#     )

In [ ]:
selected_model = models["LightGBM"]

In [ ]:
# analyzer.evaluate_feature_removal_combinations(
#     model=selected_model,
#     features=features,
#     candidates=features,
#     export_html=True,
#     file_name=f"{name.lower().replace(' ', '-')}")

In [ ]:
selected_features =  ['season', 'yr', 'mnth', 'weekday', 'temp', 'hum', 'windspeed']

In [ ]:
analyzer.compare_feature_sets(
    model=selected_model,
    feature_sets={
        "All Features": features,
        "Model 'holiday', 'workingday', 'weathersit', 'atemp'": selected_features,
    },
    export_html=True,
    file_name="lighgbm_feature_comparison",
)

In [ ]:
analyzer.residual_analysis(model=selected_model,export_html=True)

In [ ]:
analyzer.learning_curve(model=selected_model)